# Goal
Read notebook for more details:

In [1]:
import json
import matplotlib.pyplot as plt
import pandas as pd
import scanpy as sc
from pathlib import Path
import os
from dotenv import load_dotenv; load_dotenv()
import ipynbname
import shutil

# MUST BE FIRST - before any imports from cell_type_mapper --> error if FromSpecifiedMarkersRunner run wiht GPU
os.environ['CUDA_VISIBLE_DEVICES'] = ''

import cell_type_mapper
from abc_atlas_access.abc_atlas_cache.abc_project_cache import AbcProjectCache
from cell_type_mapper.cli.from_specified_markers import FromSpecifiedMarkersRunner

# Load data

Your query data must be:
- **Format**: `.h5ad` file (AnnData format)
- **Structure**: 
  - `X` layer with **RAW** gene expression data (cells × genes)
  - `obs` with cell metadata
  - `var` with gene names, **Ensembl IDs** for MapMyCells-supported taxonomies

In [2]:
TISSUE = "DFC"
QUERY_PATH = f"/home/gdallagl/myworkdir/XDP/data/XDP/NucSec/adatas/{TISSUE}/{TISSUE}_combined_QC.h5ad"

# paths to files where mapping output will be written
json_dst_path = str(Path(QUERY_PATH).parent / "map_my_cell" / "mapping.json")
csv_dst_path = str(Path(QUERY_PATH).parent / "map_my_cell" / "mapping.csv")
os.makedirs(os.path.dirname(json_dst_path), exist_ok=True)
os.makedirs(os.path.dirname(csv_dst_path), exist_ok=True)

# saving adata path
adata_labelled_path = f"{os.path.splitext(QUERY_PATH)[0]}_mmc.h5ad"
adata_labelled_path

'/home/gdallagl/myworkdir/XDP/data/XDP/NucSec/adatas/DFC/DFC_combined_QC_mmc.h5ad'

# Define Atlas/ref 

In [3]:
query_marker_path = "/home/gdallagl/myworkdir/XDP/data/AllenAtlas/BGT_human_20250507/Human.query_markers.20250507.json"
precomputed_path = "/home/gdallagl/myworkdir/XDP/data/AllenAtlas/BGT_human_20250507/Human.precomputed_stats.20250507.h5"

# Check gene overlapping

In [4]:
import json
import anndata as ad

# Load reference marker genes
with open(query_marker_path) as f:
    markers = json.load(f)

ref_genes = set()
for genes in markers.values():
    ref_genes.update(genes)

# Load your adata
adata = ad.read_h5ad(QUERY_PATH, backed="r")
query_genes = set(adata.var_names)

overlap = ref_genes & set(adata.var_names)
print(f"Reference genes: {len(ref_genes)}")
print(f"Query genes:     {len(adata.var_names)}")
print(f"Overlap:         {len(overlap)} ({len(overlap)/len(ref_genes)*100:.1f}%)")

del adata


Reference genes: 6852
Query genes:     38601
Overlap:         6514 (95.1%)


# Run Mapping

Now we will actually [perform the mapping](https://github.com/AllenInstitute/cell_type_mapper/blob/main/docs/mapping_cells.md).

In [5]:
config = {
    # output paths
    "query_path": QUERY_PATH,
    "extended_result_path": json_dst_path,
    "csv_result_path": csv_dst_path,
    "verbose_csv": True,

    # inout paths
    "query_markers": {
       "serialized_lookup": query_marker_path
    },
    "precomputed_stats": {
        "path": precomputed_path
    },

    "type_assignment": {
        "n_processors": 32,
        "normalization": "raw", # Use raw counts (not normalized)
        "bootstrap_factor": 0.5,
        "bootstrap_iteration": 100
    }
}

In [6]:
runner = FromSpecifiedMarkersRunner(
    args=[],
    input_data=config
)
runner.run()
print("Done!")

=== Running Hierarchical Mapping 1.5.2 with config ===
{
  "query_markers": {
    "collapse_markers": false,
    "serialized_lookup": "/home/gdallagl/myworkdir/XDP/data/AllenAtlas/BGT_human_20250507/Human.query_markers.20250507.json",
    "log_level": "ERROR"
  },
  "obsm_key": null,
  "map_to_ensembl": false,
  "extended_result_path": "/home/gdallagl/myworkdir/XDP/data/XDP/NucSec/adatas/DFC/map_my_cell/mapping.json",
  "precomputed_stats": {
    "log_level": "ERROR",
    "path": "/home/gdallagl/myworkdir/XDP/data/AllenAtlas/BGT_human_20250507/Human.precomputed_stats.20250507.h5"
  },
  "extended_result_dir": null,
  "csv_result_path": "/home/gdallagl/myworkdir/XDP/data/XDP/NucSec/adatas/DFC/map_my_cell/mapping.csv",
  "log_level": "ERROR",
  "cloud_safe": false,
  "flatten": false,
  "verbose_csv": true,
  "summary_metadata_path": null,
  "obsm_clobber": false,
  "query_gene_id_col": null,
  "tmp_dir": null,
  "type_assignment": {
    "bootstrap_factor": 0.5,
    "normalization": "raw

/home/gdallagl/myworkdir/XDP/.venv/lib/python3.11/site-packages/cell_type_mapper/cli/cli_log.py:73: UserWarning: numpy's internal parallelization is enabled. This could cause independent worker processes to compete for resources, degrading performance. We recommend setting the following environment variables to '1' to improve performance
{
  "NUMEXPR_NUM_THREADS": "",
  "MKL_NUM_THREADS": "",
  "OMP_NUM_THREADS": ""
}
  warnings.warn(msg)
/home/gdallagl/myworkdir/XDP/.venv/lib/python3.11/site-packages/cell_type_mapper/cli/cli_log.py:104: FutureWarning: `__version__` is deprecated, use `importlib.metadata.version('anndata')` instead.
  self.env(f"anndata version: {anndata.__version__}")
/home/gdallagl/myworkdir/XDP/.venv/lib/python3.11/site-packages/cell_type_mapper/taxonomy/utils.py:253: UserWarning: This taxonomy has no mapping from leaf_node -> rows in the cell by gene matrix
  warnings.warn("This taxonomy has no mapping from leaf_node -> rows "
/home/gdallagl/myworkdir/XDP/.venv/lib

BENCHMARK: spent 1.5799e-01 seconds creating query marker cache
Running CPU implementation of type assignment.
BENCHMARK: spent 1.1703e+03 seconds assigning cell types
Writing marker genes to output file
MAPPING FROM SPECIFIED MARKERS RAN SUCCESSFULLY
CLEANING UP
Done!


# Output of mapping file

The results of our mapping are now in two files: the csv file pointed to by `csv_dst_path` and the JSON file pointed to by `json_dst_path`. Dedicated documentation of the the contents of the mapping output [can be found here.](https://github.com/AllenInstitute/cell_type_mapper/blob/main/docs/output.md)

## CSV output file

The CSV file is effectively just a dataframe. For every cell at every taxonomy level, you have its assigned cell type (both as a guaranteed unique "label" and a more human readable "name") along with quality metrics assessing the confidence in the mapping (see the detailed documentation above).

In [7]:
mapping_csv = pd.read_csv(csv_dst_path, comment='#')
mapping_csv = mapping_csv.set_index("cell_id")

display(mapping_csv)

,Neighborhood_label,Neighborhood_name,Neighborhood_bootstrapping_probability,Neighborhood_aggregate_probability,Neighborhood_correlation_coefficient,Class_label,Class_name,Class_bootstrapping_probability,Class_aggregate_probability,Class_correlation_coefficient,...,Group_name,Group_bootstrapping_probability,Group_aggregate_probability,Group_correlation_coefficient,Cluster_label,Cluster_name,Cluster_alias,Cluster_bootstrapping_probability,Cluster_aggregate_probability,Cluster_correlation_coefficient
cell_id,,,,,,,,,,,,,,,,,,,,,
22CTCMLT4__pXDPsHSrDLFCid240830rxn4__TCAGGTAAGCGCTTAT-1_DFC_SCF-18-006,CS20250428_NEIGH_0001,Nonneuron,1.0,1.0,0.7671,CS20250428_CLASS_0000,Astro-Epen,1.0,1.0,0.8534,...,Astrocyte,1.00,1.00,0.4063,CS20250428_CLUST_0249,Human-14,Human-14,1.00,1.0000,0.5531
22CTCMLT4__pXDPsHSrDLFCid240830rxn4__ATGAGGGTCTATCCCG-1_DFC_SCF-19-009,CS20250428_NEIGH_0001,Nonneuron,1.0,1.0,0.7289,CS20250428_CLASS_0000,Astro-Epen,1.0,1.0,0.8059,...,Astrocyte,1.00,1.00,0.4906,CS20250428_CLUST_0255,Human-232,Human-232,0.99,0.9900,0.4850
22CTCMLT4__pXDPsHSrDLFCid240830rxn4__CTTGGCTTCTGTCTCG-1_DFC_SCF-22-058CF,CS20250428_NEIGH_0001,Nonneuron,1.0,1.0,0.7506,CS20250428_CLASS_0000,Astro-Epen,1.0,1.0,0.8471,...,Astrocyte,1.00,1.00,0.5222,CS20250428_CLUST_0249,Human-14,Human-14,0.95,0.9500,0.5342
22CTCMLT4__pXDPsHSrDLFCid240830rxn4__CTGTTTACATCCGCGA-1_DFC_SCF-19-009,CS20250428_NEIGH_0001,Nonneuron,1.0,1.0,0.7442,CS20250428_CLASS_0000,Astro-Epen,1.0,1.0,0.7963,...,Astrocyte,1.00,1.00,0.5659,CS20250428_CLUST_0255,Human-232,Human-232,0.59,0.5900,0.5059
22CTCMLT4__pXDPsHSrDLFCid240830rxn4__GCAAACTTCACTCCTG-1_DFC_SCF-19-014,CS20250428_NEIGH_0001,Nonneuron,1.0,1.0,0.7484,CS20250428_CLASS_0000,Astro-Epen,1.0,1.0,0.8262,...,Astrocyte,1.00,1.00,0.5211,CS20250428_CLUST_0249,Human-14,Human-14,1.00,1.0000,0.5044
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
250328_SL-EXE_0516_A22N2VFLT4__SI-TT-G6__CGGGTAGCACGGACCT-1_DFC_SCF_22-060,CS20250428_NEIGH_0001,Nonneuron,1.0,1.0,0.7447,CS20250428_CLASS_0010,OPC-Oligo,1.0,1.0,0.8398,...,Oligo OPALIN,1.00,1.00,0.6838,CS20250428_CLUST_0227,Human-1,Human-1,1.00,1.0000,0.5114
250328_SL-EXE_0516_A22N2VFLT4__SI-TT-G6__CGTGAAGAGAGGTTGG-1_DFC_SCF-19-020,CS20250428_NEIGH_0001,Nonneuron,1.0,1.0,0.7433,CS20250428_CLASS_0010,OPC-Oligo,1.0,1.0,0.8004,...,Oligo OPALIN,1.00,1.00,0.6405,CS20250428_CLUST_0227,Human-1,Human-1,1.00,1.0000,0.5902
250328_SL-EXE_0516_A22N2VFLT4__SI-TT-G6__AGGTTGGAGACAGACG-1_DFC_SCF_23-083,CS20250428_NEIGH_0001,Nonneuron,1.0,1.0,0.7445,CS20250428_CLASS_0010,OPC-Oligo,1.0,1.0,0.8479,...,Oligo OPALIN,1.00,1.00,0.5887,CS20250428_CLUST_0227,Human-1,Human-1,1.00,1.0000,0.6741


# Add Metadata to adata and Save

In [8]:
# Read adata
query_adata = sc.read_h5ad(QUERY_PATH)

# Merge metadata
query_adata.obs = query_adata.obs.join(mapping_csv, how="left")
display(query_adata.obs)

# save 
query_adata.write(adata_labelled_path)


n_cells_original = len(query_adata.obs)
n_cells_mapped = mapping_csv.index.nunique()
print(f"Original cells: {n_cells_original}")
print(f"Mapped cells: {n_cells_mapped}")
assert n_cells_original == n_cells_mapped, "Mapping incomplete!"

,background_fraction,cell_probability,cell_size,droplet_efficiency,barcode,bcl,rna_index,library,library__barcode,frac_mito,...,Group_name,Group_bootstrapping_probability,Group_aggregate_probability,Group_correlation_coefficient,Cluster_label,Cluster_name,Cluster_alias,Cluster_bootstrapping_probability,Cluster_aggregate_probability,Cluster_correlation_coefficient
barcode,,,,,,,,,,,,,,,,,,,,,
22CTCMLT4__pXDPsHSrDLFCid240830rxn4__TCAGGTAAGCGCTTAT-1_DFC_SCF-18-006,0.013631,0.999955,12522.672852,1.630931,22CTCMLT4__pXDPsHSrDLFCid240830rxn4__TCAGGTAAG...,22CTCMLT4,pXDPsHSrDLFCid240830rxn4,22CTCMLT4__pXDPsHSrDLFCid240830rxn4,22CTCMLT4__pXDPsHSrDLFCid240830rxn4__TCAGGTAAG...,0.000385,...,Astrocyte,1.00,1.00,0.4063,CS20250428_CLUST_0249,Human-14,Human-14,1.00,1.0000,0.5531
22CTCMLT4__pXDPsHSrDLFCid240830rxn4__ATGAGGGTCTATCCCG-1_DFC_SCF-19-009,0.012919,0.999955,12225.633789,1.590550,22CTCMLT4__pXDPsHSrDLFCid240830rxn4__ATGAGGGTC...,22CTCMLT4,pXDPsHSrDLFCid240830rxn4,22CTCMLT4__pXDPsHSrDLFCid240830rxn4,22CTCMLT4__pXDPsHSrDLFCid240830rxn4__ATGAGGGTC...,0.000174,...,Astrocyte,1.00,1.00,0.4906,CS20250428_CLUST_0255,Human-232,Human-232,0.99,0.9900,0.4850
22CTCMLT4__pXDPsHSrDLFCid240830rxn4__CTTGGCTTCTGTCTCG-1_DFC_SCF-22-058CF,0.017131,0.999955,12366.195312,1.381835,22CTCMLT4__pXDPsHSrDLFCid240830rxn4__CTTGGCTTC...,22CTCMLT4,pXDPsHSrDLFCid240830rxn4,22CTCMLT4__pXDPsHSrDLFCid240830rxn4,22CTCMLT4__pXDPsHSrDLFCid240830rxn4__CTTGGCTTC...,0.000132,...,Astrocyte,1.00,1.00,0.5222,CS20250428_CLUST_0249,Human-14,Human-14,0.95,0.9500,0.5342
22CTCMLT4__pXDPsHSrDLFCid240830rxn4__CTGTTTACATCCGCGA-1_DFC_SCF-19-009,0.013656,0.999955,12066.066406,1.410911,22CTCMLT4__pXDPsHSrDLFCid240830rxn4__CTGTTTACA...,22CTCMLT4,pXDPsHSrDLFCid240830rxn4,22CTCMLT4__pXDPsHSrDLFCid240830rxn4,22CTCMLT4__pXDPsHSrDLFCid240830rxn4__CTGTTTACA...,0.000000,...,Astrocyte,1.00,1.00,0.5659,CS20250428_CLUST_0255,Human-232,Human-232,0.59,0.5900,0.5059
22CTCMLT4__pXDPsHSrDLFCid240830rxn4__GCAAACTTCACTCCTG-1_DFC_SCF-19-014,0.013716,0.999955,11708.838867,1.452997,22CTCMLT4__pXDPsHSrDLFCid240830rxn4__GCAAACTTC...,22CTCMLT4,pXDPsHSrDLFCid240830rxn4,22CTCMLT4__pXDPsHSrDLFCid240830rxn4,22CTCMLT4__pXDPsHSrDLFCid240830rxn4__GCAAACTTC...,0.000200,...,Astrocyte,1.00,1.00,0.5211,CS20250428_CLUST_0249,Human-14,Human-14,1.00,1.0000,0.5044
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
250328_SL-EXE_0516_A22N2VFLT4__SI-TT-G6__CGGGTAGCACGGACCT-1_DFC_SCF_22-060,0.398194,0.995271,10058.555664,0.599375,250328_SL-EXE_0516_A22N2VFLT4__SI-TT-G6__CGGGT...,250328_SL-EXE_0516_A22N2VFLT4,SI-TT-G6,250328_SL-EXE_0516_A22N2VFLT4__SI-TT-G6,250328_SL-EXE_0516_A22N2VFLT4__SI-TT-G6__CGGGT...,0.000000,...,Oligo OPALIN,1.00,1.00,0.6838,CS20250428_CLUST_0227,Human-1,Human-1,1.00,1.0000,0.5114
250328_SL-EXE_0516_A22N2VFLT4__SI-TT-G6__CGTGAAGAGAGGTTGG-1_DFC_SCF-19-020,0.375699,0.994088,10292.061523,0.569160,250328_SL-EXE_0516_A22N2VFLT4__SI-TT-G6__CGTGA...,250328_SL-EXE_0516_A22N2VFLT4,SI-TT-G6,250328_SL-EXE_0516_A22N2VFLT4__SI-TT-G6,250328_SL-EXE_0516_A22N2VFLT4__SI-TT-G6__CGTGA...,0.000000,...,Oligo OPALIN,1.00,1.00,0.6405,CS20250428_CLUST_0227,Human-1,Human-1,1.00,1.0000,0.5902
250328_SL-EXE_0516_A22N2VFLT4__SI-TT-G6__AGGTTGGAGACAGACG-1_DFC_SCF_23-083,0.449985,0.998483,9964.245117,0.594584,250328_SL-EXE_0516_A22N2VFLT4__SI-TT-G6__AGGTT...,250328_SL-EXE_0516_A22N2VFLT4,SI-TT-G6,250328_SL-EXE_0516_A22N2VFLT4__SI-TT-G6,250328_SL-EXE_0516_A22N2VFLT4__SI-TT-G6__AGGTT...,0.000000,...,Oligo OPALIN,1.00,1.00,0.5887,CS20250428_CLUST_0227,Human-1,Human-1,1.00,1.0000,0.6741


Original cells: 145741
Mapped cells: 145741


In [9]:
# print(query_adata.obs.columns)#Group_names.value_counts()
# query_adata.obs.supercluster_name.value_counts()